### RAG Pipeline Data Ingestion -> Vector DB


In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_111209/972835014.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/home/seeker/Projects/VectorRAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = os.getenv("NVIDIA_API_KEY")

HF_TOKEN = os.getenv("HF_TOKEN")

EVAL_API_KEY = os.getenv("EVAL_API_KEY")

In [3]:
def process_all_pdf(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:  # noqa: BLE001
            print(f"Error: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdf("../Policy Documents Curated 15/")


Found 15 PDF Files to process

Processing: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf
Loaded 14 pages

Processing: Click-2-Protect-Optima-Secure-Policy-Bond-101Y122V05.pdf
Loaded 43 pages

Processing: hdfc-life-smart-pension-plus-v13-policy-document-individual.pdf
Loaded 42 pages

Processing: Bharat_Griha_Raksha_Policy_Policy_Wordings_5219f40e18.pdf
Loaded 21 pages

Processing: Auto_Secure_Commercial_Vehicle_Package_Policy_Base_Policy_Wording_22b7905015.pdf
Loaded 47 pages

Processing: Cyber_Shield_Policy_Wordings_78baa23b5a.pdf
Loaded 11 pages

Processing: tata_aig_travel_insurance_international_plus_health_policy_wordings_012c2ec139.pdf
Loaded 62 pages

Processing: Weather_Insurance_Policy_Wordings_Retail_6a1bd806a7.pdf
Loaded 13 pages

Processing: SBI_General_Livestock_Policy_Wording_38ef0b201b.pdf
Loaded 5 pages

Processing: 14..Trade credit insc_GEN756.pdf
Loaded 20 pages

Processing: PORT_PACKAGE_Policy_wording_a5f317091b.pdf
Loaded 165 pages

Processing: Policy_Wordings

In [4]:
from pathlib import Path

directory = Path("../Policy Documents Curated 15")
pdf_count = sum(1 for file in directory.glob("*.pdf"))

print(f"Total PDFs: {pdf_count}")


Total PDFs: 15


### Text Splitting get into chunks


In [5]:
def split_documents(
    documents,
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
    min_chunk_size: int = 80,
) -> list:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=[
            "\n\n\n",  # Section breaks
            "\n\n",  # Paragraph breaks
            "\n",  # Numbered lists / clause lines
            ". ",  # Sentence boundaries
            "? ",
            "! ",
            "; ",  # Legal list semicolons
            ", ",
            " ",
            "",
        ],
        length_function=len,
        is_separator_regex=False,
        keep_separator=True,
    )

    split_docs = text_splitter.split_documents(documents)
    original_count = len(split_docs)

    cleaned_docs = []
    for doc in split_docs:
        stripped_text = doc.page_content.strip()

        # Filter 1: Drop chunks below min character threshold
        if len(stripped_text) < min_chunk_size:
            continue

        # Filter 2: Drop chunks that are mostly non-alphanumeric junk
        alnum_chars = sum(c.isalnum() for c in stripped_text)
        if (alnum_chars / max(len(stripped_text), 1)) < 0.3:
            continue

        # Filter 3: Drop exact single-line header/footer noise
        lines = [line.strip() for line in stripped_text.split("\n") if line.strip()]
        if len(lines) == 1 and any(
            lines[0].lower().startswith(p) for p in ["page ", "www.", "copyright"]
        ):
            continue

        # Use cleaned text, not raw
        doc.page_content = stripped_text
        cleaned_docs.append(doc)

    # Enrich metadata
    for i, doc in enumerate(cleaned_docs):
        doc.metadata["chunk_index"] = i
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["chunk_word_count"] = len(doc.page_content.split())

    print(f"Split {len(documents)} documents into {original_count} chunks")
    print(f"After filtering: {len(cleaned_docs)} chunks")
    print(f"Filtered out: {original_count - len(cleaned_docs)} noisy/tiny chunks")

    if cleaned_docs:
        sizes = [len(d.page_content) for d in cleaned_docs]
        print("\nChunk size stats:")
        print(f"  Min: {min(sizes)} chars")
        print(f"  Max: {max(sizes)} chars")
        print(f"  Avg: {sum(sizes) // len(sizes)} chars")
        print("\nExample chunk preview:")
        print(f"  Content: {cleaned_docs[0].page_content[:200]}...")
        print(f"  Metadata: {cleaned_docs[0].metadata}")

    return cleaned_docs


### Caching the chunks


In [6]:
import pickle
from pathlib import Path

CACHE_DIR = Path("../cache")
CACHE_DIR.mkdir(exist_ok=True)
chunks_file = CACHE_DIR / "chunks.pkl"

if chunks_file.exists():
    with open(chunks_file, "rb") as f:
        chunks = pickle.load(f)
    print("Loaded cached chunks from disk")
else:
    chunks = split_documents(all_pdf_documents)
    with open(chunks_file, "wb") as f:
        pickle.dump(chunks, f)
    print("Saved chunks to disk")


Loaded cached chunks from disk


### Embedding


In [7]:
import uuid

import chromadb
import numpy as np
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

In [8]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings


class EmbeddingManager:
    def __init__(
        self,
        model_name: str = "nvidia/nv-embed-v1",
        api_key: str | None = API_KEY,
        base_url: str = "https://integrate.api.nvidia.com/v1",
    ):
        self.model_name = model_name
        self.api_key = api_key
        self.base_url = base_url
        self._embedding_dimension: int | None = None

        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = NVIDIAEmbeddings(
                model=self.model_name,
                nvidia_api_key=self.api_key,
                base_url=self.base_url,
                max_batch_size=20,
                truncate="END",
            )
            self.model._client.timeout = 120
            print("Model loaded successfully.")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(
        self, texts: list[str], batch_size: int = 20, max_retries: int = 3
    ) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        import time

        all_embeddings = []
        total = len(texts)
        print(f"Generating embeddings for {total} texts in batches of {batch_size}...")

        for i in range(0, total, batch_size):
            batch = texts[i : i + batch_size]
            batch_num = i // batch_size + 1
            total_batches = (total + batch_size - 1) // batch_size
            for attempt in range(1, max_retries + 1):
                try:
                    print(f"  Batch {batch_num}/{total_batches} (attempt {attempt})...")
                    result = self.model.embed_documents(batch)
                    all_embeddings.extend(result)
                    print(
                        f"  Batch {batch_num}/{total_batches} done ({len(all_embeddings)}/{total} total)"
                    )
                    break
                except Exception as e:
                    print(f"  Batch {batch_num} attempt {attempt} failed: {e}")
                    if attempt < max_retries:
                        wait = 2**attempt
                        print(f"  Retrying in {wait}s...")
                        time.sleep(wait)
                    else:
                        raise

        print(f"Generated embeddings count: {len(all_embeddings)}")
        return np.asarray(all_embeddings)

    def get_embedding_dimension(self) -> int:
        if self.model is None:
            raise ValueError("Model not loaded")

        if self._embedding_dimension is None:
            sample = self.model.embed_query("probe")
            self._embedding_dimension = len(sample)

        return self._embedding_dimension

    def generate_query_embedding(self, query: str) -> np.ndarray:
        embedding = self.model.embed_query(query)
        return np.asarray(embedding)


In [9]:
embedding_manager = EmbeddingManager(api_key=API_KEY)

Loading embedding model: nvidia/nv-embed-v1
Model loaded successfully.


### Vector Store


In [10]:
import os
from typing import Any

import chromadb  # noqa: F811
import numpy as np  # noqa: F811


class VectorStore:
    def __init__(
        self, collection_name: str, persist_directory: str = "../vector_store/"
    ) -> None:
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF Document embeddings for RAG",
                    "hnsw:space": "cosine",
                },
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Exception intializing vector store: {e}")
            raise

    def add_documents(self, documents: list[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vectore store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            assert self.collection is not None
            batch_size = 1000
            for start in range(0, len(documents), batch_size):
                end = start + batch_size
                self.collection.add(
                    ids=ids[start:end],
                    embeddings=embeddings_list[start:end],
                    metadatas=metadatas[start:end],
                    documents=documents_text[start:end],
                )
            print(f"Successfully added {len(documents)} documents to vector DB")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# vector_store = VectorStore(collection_name="policy_documents", persist_directory="../vectorDB")
# vector_store.add_documents(embeddings=embeddings, documents=chunks)


In [11]:
import pickle
from pathlib import Path

import numpy as np

CACHE_DIR = Path("../cache")
CACHE_DIR.mkdir(exist_ok=True)

chunks_file = CACHE_DIR / "chunks.pkl"
embeddings_file = CACHE_DIR / "embeddings.npy"

# chunks
if chunks_file.exists():
    with open(chunks_file, "rb") as f:
        chunks = pickle.load(f)
    print(f"✓ Loaded {len(chunks)} chunks from cache")
else:
    chunks = split_documents(all_pdf_documents)
    with open(chunks_file, "wb") as f:
        pickle.dump(chunks, f)
    print(f"✓ Saved {len(chunks)} chunks to cache")

# embeddings
if embeddings_file.exists():
    embeddings = np.load(embeddings_file)
    print(f"✓ Loaded embeddings from cache: {embeddings.shape}")
else:
    embeddings = embedding_manager.generate_embeddings(
        [doc.page_content for doc in chunks]
    )
    np.save(embeddings_file, embeddings)
    print(f"✓ Saved embeddings to cache: {embeddings.shape}")

# vectorDB
vector_store = VectorStore(
    collection_name="policy_documents",
    persist_directory="../vectorDB",
)
if vector_store.collection.count() == 0:
    vector_store.add_documents(embeddings=embeddings, documents=chunks)
    print("✓ Vector DB populated and persisted")
else:
    print(
        f"✓ Vector DB already has {vector_store.collection.count()} docs — skipping ingestion"
    )


✓ Loaded 2096 chunks from cache
✓ Loaded embeddings from cache: (2096, 4096)
Vector store initialized. Collection: policy_documents
Existing documents in collection: 2096
✓ Vector DB already has 2096 docs — skipping ingestion


In [12]:
import os
from pathlib import Path

import chromadb

vector_db_dir = Path("../vectorDB")

if vector_db_dir.exists() and any(vector_db_dir.iterdir()):
    print("Using existing vector DB")
else:
    print("Building vector DB from scratch")
    vector_store = VectorStore(
        collection_name="policy_documents", persist_directory="../vectorDB"
    )
    embeddings = embedding_manager.generate_embeddings(
        [doc.page_content for doc in chunks]
    )
    vector_store.add_documents(embeddings=embeddings, documents=chunks)


Using existing vector DB


### Retriever Pipeline From VectorStore


In [13]:
class RAGRetriever:
    def __init__(
        self, vector_store: VectorStore, embedding_manager: EmbeddingManager
    ) -> None:
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> list[dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            if self.vector_store is None or self.vector_store.collection is None:
                raise ValueError("Vector store collection is not initialized.")

            query_embedding = self.embedding_manager.generate_query_embedding(query)

            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents: list[str] = results["documents"][0]
                metadatas: list[dict] = results["metadatas"][0]  # type: ignore
                distances: list[float] = results["distances"][0]  # type: ignore
                ids: list[str] = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = (
                        1 - distance
                    )  # ← ChromaDB cosine distance = 1 - similarity

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "rank": i + 1,
                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:  # noqa: BLE001
            print(f"Error retrieving documents: {e}")
            return []

In [14]:
if "vector_store" not in globals():
    print("Initializing vector_store because it was not defined in this session.")
    vector_store = VectorStore(
        collection_name="policy_documents",
        persist_directory="../vectorDB",
    )

rag_retriever = RAGRetriever(
    vector_store=vector_store,
    embedding_manager=embedding_manager,
)


In [15]:
rag_retriever.retrieve("what are the benefits of a health insurance?")

Retrieving documents for query: what are the benefits of a health insurance?
Top K: 5, Score threshold: 0.0
Retrieved 5 documents (after filtering)


[{'id': 'doc_b2d62af2_26',
  'content': 'per plan opted) from the commencement of a health \ninsurance policy during which period speci\x1fed diseases/ \ntreatments (except due to an accident) are not covered. On \ncompletion of the period, diseases/treatments shall be \ncovered provided the policy has been continuously renewed \nwithout any break.\n• \nSolicitation means the act of approaching a prospect or a \nPolicyholder by an Insurer or by a distribution channel with a \nview to persuading the prospect or a Policyholder to purchase \nor to renew an insurance Policy.\n• \nSub-limit means a cost sharing requirement under a health \ninsurance policy in which an insurer would not be liable to pay \nany amount in excess of the pre-deﬁned limit\n• \nSum Insured means the pre-deﬁned limit speciﬁed in the \nPolicy Schedule. Sum Insured and Cumulative Bonus \nrepresents the maximum, total and cumulative liability for any \nand all claims made under the Policy, in respect of that Insured',


### Integration VectorDB Context Pipeline With LLM Output


In [16]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
    model="nvidia/nemotron-3-nano-30b-a3b",
    nvidia_api_key=API_KEY,
    temperature=0.0,
    max_completion_tokens=1024,
)


In [17]:
classify_llm = ChatNVIDIA(
    model="meta/llama-3.1-8b-instruct",
    nvidia_api_key=API_KEY,
    temperature=0.0,
    max_completion_tokens=1024,
)


In [18]:
GENERAL_SYSTEM_PROMPT = """You are an expert insurance advisor.
Answer the following general insurance question clearly and helpfully.
Use your general knowledge — you do NOT need to reference any specific policy document.

Question: {query}

Answer:"""


SPECIFIC_SYSTEM_PROMPT = """You are an expert insurance policy assistant. Answer the question using ONLY the provided context from the actual policy documents.

Rules:
- Use the context to provide a direct, factual answer
- If the context partially answers the question, provide what you can and note what is missing
- Only say "I don't have enough information" if the context is completely irrelevant to the question
- Be concise and precise
- Do not hallucinate or add information not present in the context"""


CLASSIFIER_PROMPT = """You are a query classifier for an insurance policy assistant.

Classify the query into exactly one of these categories:
- general: insurance-related conceptual, definitional, or comparative questions (what is health insurance?, explain copay, how does deductible work, difference between HMO and PPO). Must be about insurance or a related financial/health topic.
- specific: the user asks about a concrete provision, condition, claim, or coverage detail of a policy — the kind of thing you'd only find in the actual wording (grace periods, exclusions, claim process, deductibles, coverage limits).
- ambiguous: too vague to route (e.g. tell me about my policy).
- irrelevant: NOT related to insurance, policies, coverage, health plans, or financial protection at all (e.g. programming questions, cooking recipes, general trivia, math problems).

Respond with ONLY the category word (general, specific, ambiguous, or irrelevant), nothing else.

Query: {query}
Category:"""

In [19]:
import time

from requests.exceptions import ReadTimeout


def classify_query(query: str, llm: ChatNVIDIA = classify_llm, retries: int = 3) -> str:
    """Classify using classify_llm ONLY. Returns general, specific, or ambiguous."""
    for attempt in range(retries):
        try:
            response = llm.invoke(CLASSIFIER_PROMPT.format(query=query))
            category = response.content.strip().lower()
            if category not in ("general", "specific", "ambiguous"):
                # default to specific so we attempt retrieval
                return "specific"
            return category
        except ReadTimeout:
            if attempt < retries - 1:
                print(f"Timeout on attempt {attempt + 1}, retrying...")
                time.sleep(2)
            else:
                print("Classifier timed out, defaulting to specific")
                return "specific"


In [20]:
from requests.exceptions import ReadTimeout


def rag_query(
    query: str,
    retriever: RAGRetriever = rag_retriever,
    llm: ChatNVIDIA = llm,
    top_k: int = 5,
    score_threshold: float = 0.2,
) -> dict[str, Any]:

    # guard empty query
    if not query or not query.strip():
        return {"answer": "Query cannot be empty", "sources": [], "context_used": False}

    # step 1 — classify using classify_llm ONLY
    category = classify_query(query)

    # step 1b — irrelevant: refuse to answer
    if category == "irrelevant":
        return {
            "answer": "I'm an insurance policy assistant and can only help with questions related to insurance, policies, coverage, or health plans. Please ask a relevant question.",
            "sources": [],
            "context_used": False,
            "category": category,
            "confidence": 1.0,
        }

    # step 2a — general: answer from LLM knowledge, no retrieval
    if category == "general":
        for attempt in range(3):
            try:
                response = llm.invoke(GENERAL_SYSTEM_PROMPT.format(query=query))
                return {
                    "answer": response.content,
                    "sources": [],
                    "context_used": False,
                    "category": category,
                    "confidence": 1.0,
                }
            except ReadTimeout:
                if attempt < 2:
                    print(f"General answer timeout, retrying attempt {attempt + 2}...")
                    time.sleep(3)
                else:
                    return {
                        "answer": "Generation timed out. Please try again.",
                        "sources": [],
                        "context_used": False,
                        "category": category,
                        "confidence": 0.0,
                    }

    # step 2b — ambiguous: ask for clarification
    if category == "ambiguous":
        return {
            "answer": "Your question is too vague. Could you specify which policy or coverage detail you're asking about?",
            "sources": [],
            "context_used": False,
            "category": category,
        }

    # step 2c — specific: retrieve from vector store
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)

    if not results:
        return {
            "answer": "I don't have enough information in the provided context to answer this.",
            "sources": [],
            "context_used": False,
            "category": category,
            "confidence": 0.0,
        }

    # step 3 — build context
    context_parts = []
    sources = []
    for i, doc in enumerate(results):
        context_parts.append(f"[{i + 1}] {doc['content']}")
        sources.append(
            {
                "id": doc["id"],
                "score": round(doc["similarity_score"], 4),
                "page": doc["metadata"].get("page", "unknown"),
                "source": doc["metadata"].get(
                    "source_file", doc["metadata"].get("source", "unknown")
                ),
                "preview": doc["content"][:150].strip() + "...",
            }
        )

    confidence = max([doc["similarity_score"] for doc in results])
    context = "\n\n".join(context_parts)

    prompt = f"""{SPECIFIC_SYSTEM_PROMPT}

    Context: {context}

    Question: {query}

    Answer:"""

    # step 4 — generate
    for attempt in range(3):
        try:
            response = llm.invoke(prompt)
            return {
                "answer": response.content,
                "sources": sources,
                "context_used": True,
                "retrieved_count": len(results),
                "category": category,
                "confidence": confidence,
            }
        except ReadTimeout:
            if attempt < 2:
                print(f"Generation timeout, retrying attempt {attempt + 2}...")
                time.sleep(3)
            else:
                return {
                    "answer": "Generation timed out. Please try again.",
                    "sources": sources,
                    "context_used": False,
                    "category": category,
                    "confidence": 0.0,
                }


In [21]:
query1 = "Why should i have a health insurance"
query2 = "Write me a python program for Hello World"
query3 = "Under what timeframe must the Insured submit a complete written claim to the Insurer after the Date of Loss for the claim to be payable?"
output = rag_query(query=query1, top_k=3)
# print(output)
# print(type(output))
for key, value in output.items():
    print(f"{key}: {value}\n")

answer: **Why having health insurance is essential**

| Reason | What it means for you | Why it matters |
|--------|----------------------|----------------|
| **Financial protection** | Shields you from potentially catastrophic medical bills. | A single hospital stay or surgery can cost tens of thousands of dollars. Without insurance, you’d have to pay that out‑of‑pocket, which can quickly deplete savings or force you into debt. |
| **Access to care** | Gives you a network of doctors, hospitals, specialists, and prescription drug plans. | You can see a physician when you’re sick, get timely treatment for chronic conditions, and receive preventive services (vaccines, screenings) that keep you healthier overall. |
| **Preventive and wellness services** | Most plans cover annual check‑ups, immunizations, cancer screenings, and lifestyle counseling at little or no cost. | Early detection and regular monitoring can prevent serious illnesses, reduce long‑term treatment costs, and improve qua

In [22]:
output = rag_query(query=query2, top_k=3)
for key, value in output.items():
    print(f"{key}: {value}\n")

Retrieving documents for query: Write me a python program for Hello World
Top K: 3, Score threshold: 0.2
Retrieved 0 documents (after filtering)
answer: I don't have enough information in the provided context to answer this.

sources: []

context_used: False

category: specific

confidence: 0.0



In [23]:
output = rag_query(query=query3, top_k=3)
for key, value in output.items():
    print(f"{key}: {value}\n")


Retrieving documents for query: Under what timeframe must the Insured submit a complete written claim to the Insurer after the Date of Loss for the claim to be payable?
Top K: 3, Score threshold: 0.2
Retrieved 3 documents (after filtering)
answer: The Insured must submit a complete written claim **within 60 days after the Date of Loss**.

sources: [{'id': 'doc_9d63ce9a_1092', 'score': 0.5665, 'page': 6, 'source': '14..Trade credit insc_GEN756.pdf', 'preview': '7.3. Should the Insured refuse to exercise its rights against any party responsible for the loss indemnified by \nthe Insurer, or use of such rights ha...'}, {'id': 'doc_f4614c39_936', 'score': 0.5192, 'page': 47, 'source': 'tata_aig_travel_insurance_international_plus_health_policy_wordings_012c2ec139.pdf', 'preview': 'We may require for ﬁling proofs of loss.\nC. \nClaim Forms: \nCompleted claim forms and written evidence of loss must be furnished to the Assistance \nCom...'}, {'id': 'doc_7a6362e4_446', 'score': 0.5122, 'page': 

### RAG Evaluation


In [24]:
chunks_file = CACHE_DIR / "chunks.pkl"

if not chunks_file.exists():
    raise FileNotFoundError(
        f"Chunks cache not found at {chunks_file}.\n"
        "Run RAG_pipeline.ipynb first to build and cache the chunks."
    )

with open(chunks_file, "rb") as f:
    chunks = pickle.load(f)

print(f"✅ Loaded {len(chunks)} chunks from cache")
print(f"   Sample chunk: {chunks[0].page_content[:120]}...")
print(f"   From file: {chunks[0].metadata.get('source_file', 'unknown')}")

✅ Loaded 2096 chunks from cache
   Sample chunk: POLICY WORDING
Arogya Sanjeevani Policy, SBI General Insurance Company Limited
3)   DEFINITIONS
1) PREAMBLE
This Policy ...
   From file: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf


In [25]:
import json
import os
import pickle
import random
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
EVAL_API_KEY = os.getenv("EVAL_API_KEY")

# ── Single NIM client — used for EVERYTHING in this notebook ─────────────────
nim = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=EVAL_API_KEY,
)

GENERATOR_MODEL = "meta/llama-3.1-8b-instruct"  # for test-set generation
JUDGE_MODEL = "meta/llama-3.1-70b-instruct"  # for evaluation scoring
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: If you have access to qwen/qwq-32b, set JUDGE_MODEL to that.
# A bigger judge = more reliable scores. Generator model can stay small.
# ─────────────────────────────────────────────────────────────────────────────

CACHE_DIR = Path("../cache")
CACHE_DIR.mkdir(exist_ok=True)


def nim_call(prompt: str, model: str = GENERATOR_MODEL, max_tokens: int = 512) -> str:
    """
    Single NIM API call with retry logic.
    Sleeps 1s between calls automatically to avoid 429s on free tier.
    """
    for attempt in range(3):
        try:
            time.sleep(1)  # Rate limit buffer — DO NOT REMOVE
            response = nim.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  ⚠ Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return ""  # Return empty string on total failure — handled downstream


print("✅ Setup complete.")


✅ Setup complete.


In [26]:
def generate_test_set(
    chunks,
    questions_per_pdf: int = 2,
    batch_size: int = 5,
    batch_pause: int = 10,
    seed: int = 42,
) -> list[dict]:
    """
    Generates question-answer pairs from chunks, 2 per PDF.

    Processes in batches of `batch_size` with a `batch_pause` second
    sleep between batches to avoid NVIDIA API rate limits.
    """
    random.seed(seed)

    # Group chunks by source PDF
    from collections import defaultdict

    pdf_chunks: dict[str, list] = defaultdict(list)
    for c in chunks:
        if len(c.page_content) > 200:
            src = c.metadata.get("source_file", "unknown")
            pdf_chunks[src].append(c)

    # Sample 2 chunks from each PDF
    selected = []
    for src, pool in pdf_chunks.items():
        pick = random.sample(pool, min(questions_per_pdf, len(pool)))
        selected.extend(pick)

    random.shuffle(selected)
    total = len(selected)
    print(
        f"Selected {total} chunks from {len(pdf_chunks)} PDFs ({questions_per_pdf} each)\n"
    )

    test_set = []
    batch_num = 0

    for i, chunk in enumerate(selected, 1):
        # Pause between batches
        if i > 1 and (i - 1) % batch_size == 0:
            batch_num += 1
            print(
                f"\n⏸  Batch complete. Sleeping {batch_pause}s to avoid rate limits...\n"
            )
            time.sleep(batch_pause)

        src_file = chunk.metadata.get("source_file", "?")
        page = chunk.metadata.get("page", "?")
        print(f"[{i}/{total}] Source: {src_file} p.{page}")

        prompt = f"""You are an insurance policy expert helping build a test set.

Read the policy text below carefully. Then:
1. Write ONE specific question that a real policyholder would ask about this text.
   - The question must be answerable from this text alone
   - Ask about a concrete detail: a number, a condition, a process, or a coverage rule
   - Do NOT ask vague questions like "what is insurance?"

2. Write the correct answer to that question.
   - Use only information from the text below
   - Be specific and complete (1-3 sentences)
   - Do not add information not present in the text

Policy text:
{chunk.page_content}

Respond in this EXACT JSON format (no extra text, no markdown):
{{"question": "...", "answer": "..."}}"""

        raw = nim_call(prompt, model=GENERATOR_MODEL, max_tokens=300)

        try:
            raw_clean = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(raw_clean)
            q = parsed.get("question", "").strip()
            a = parsed.get("answer", "").strip()

            if len(q) > 15 and len(a) > 20:
                test_set.append(
                    {
                        "question": q,
                        "ground_truth": a,
                        "source_chunk": chunk.page_content,
                        "source_file": src_file,
                        "page": page,
                    }
                )
                print(f"  ✅ Q: {q[:80]}...")
            else:
                print(f"  ⚠ Bad parse, skipping")

        except json.JSONDecodeError:
            print(f"  ⚠ JSON decode failed. Raw: {raw[:80]}")

    print(f"\n✅ Generated {len(test_set)} test pairs")
    return test_set


# ── Run once, cache forever ───────────────────────────────────────────────────
testset_file = CACHE_DIR / "eval_testset.json"

if testset_file.exists():
    with open(testset_file) as f:
        test_set = json.load(f)
    print(f"✅ Loaded {len(test_set)} cached test pairs — skipping generation")
else:
    test_set = generate_test_set(
        chunks, questions_per_pdf=2, batch_size=5, batch_pause=10
    )
    with open(testset_file, "w") as f:
        json.dump(test_set, f, indent=2)
    print(f"\n✅ Saved to {testset_file}")


✅ Loaded 29 cached test pairs — skipping generation


In [27]:
with open("../cache/eval_testset.json", "r", encoding="utf-8") as f:
    test_set = json.load(f)

print(len(test_set))

29


In [28]:
def run_vector_rag_pipeline(test_set: list[dict]) -> list[dict]:
    """
    Runs Vector RAG on every question and collects outputs.
    """
    outputs = []
    print(f"Running Vector RAG on {len(test_set)} questions...\n")

    for i, item in enumerate(test_set, 1):
        q = item["question"]
        print(f"[{i}/{len(test_set)}] {q[:70]}...")

        try:
            # Direct retrieval for contexts
            retrieved = rag_retriever.retrieve(q, top_k=5, score_threshold=0.2)
            contexts = [doc["content"] for doc in retrieved]

            # Full pipeline for the answer
            result = rag_query(query=q, retriever=rag_retriever, llm=llm, top_k=5)
            answer = result.get("answer", "")
            category = result.get("category", "RELEVANT")

            outputs.append(
                {
                    "question": q,
                    "ground_truth": item["ground_truth"],
                    "contexts": contexts,
                    "answer": answer,
                    "category": category,
                    "source_file": item["source_file"],
                }
            )
            print(f"  ✅ Answer: {answer[:80]}...")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            outputs.append(
                {
                    "question": q,
                    "ground_truth": item["ground_truth"],
                    "contexts": [],
                    "answer": f"ERROR: {e}",
                    "category": "ERROR",
                    "source_file": item["source_file"],
                }
            )

        time.sleep(1)  # Be gentle with NIM free tier

    return outputs


# ── Run and cache ─────────────────────────────────────────────────────────────
vector_rag_outputs_file = CACHE_DIR / "vector_rag_outputs.json"

if vector_rag_outputs_file.exists():
    with open(vector_rag_outputs_file) as f:
        vector_rag_outputs = json.load(f)
    print(f"✅ Loaded cached Vector RAG outputs ({len(vector_rag_outputs)} entries)")
else:
    vector_rag_outputs = run_vector_rag_pipeline(test_set)
    with open(vector_rag_outputs_file, "w") as f:
        json.dump(vector_rag_outputs, f, indent=2)
    print(f"✅ Saved Vector RAG outputs to {vector_rag_outputs_file}")


✅ Loaded cached Vector RAG outputs (29 entries)


In [29]:
# ── The 4 evaluation prompts ──────────────────────────────────────────────────
# Each returns JSON: {"score": float, "reasoning": str}

FAITHFULNESS_PROMPT = """You are an expert evaluator assessing whether an AI answer is faithful to its source context.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

TASK:
1. List every factual claim made in the Generated Answer.
2. For each claim, determine if it is directly supported by the Retrieved Context.
3. Score = (number of supported claims) / (total claims). If there are no claims, score 1.0.

Respond in this EXACT JSON format only (no markdown, no extra text):
{{"score": 0.0, "reasoning": "claim 1: supported/not supported because... claim 2: ..."}}

Score must be between 0.0 and 1.0."""


ANSWER_RELEVANCY_PROMPT = """You are an expert evaluator assessing whether an AI answer is relevant to the question asked.

QUESTION: {question}

GENERATED ANSWER:
{answer}

TASK:
Score how directly and completely the answer addresses the question.
- 1.0 = answer directly addresses all parts of the question
- 0.7 = answer mostly relevant but misses a part or adds off-topic content
- 0.4 = answer is vaguely related but does not really answer the question
- 0.0 = answer is completely off-topic or refuses to answer

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "explanation of why this score was given"}}"""


CONTEXT_PRECISION_PROMPT = """You are an expert evaluator assessing the quality of retrieved context for a RAG system.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT CHUNKS:
{context_numbered}

TASK:
For each retrieved chunk, decide if it is relevant to answering the question (given what the ground truth says).
Score = (number of relevant chunks) / (total chunks).
If no chunks were retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Chunk 1: relevant/not relevant because... Chunk 2: ..."}}"""


CONTEXT_RECALL_PROMPT = """You are an expert evaluator assessing whether a RAG system retrieved all necessary information.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT:
{context}

TASK:
1. List every key piece of information in the Ground Truth Answer.
2. For each key piece, check if it is present in the Retrieved Context.
3. Score = (pieces present in context) / (total key pieces).
If no context was retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Key point 1: found/not found in context... Key point 2: ..."}}"""


print("✅ Evaluation prompts defined.")


✅ Evaluation prompts defined.


In [30]:
def safe_parse_score(raw: str) -> dict:
    """
    Parses NIM's JSON response robustly.
    Handles cases where the model adds markdown fences or extra text.
    Returns {"score": float, "reasoning": str} or a fallback.
    """
    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        # Find the JSON object even if there's trailing text
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start == -1 or end == 0:
            raise ValueError("No JSON object found")
        parsed = json.loads(clean[start:end])
        score = float(parsed.get("score", 0.0))
        score = max(0.0, min(1.0, score))  # Clamp to [0, 1]
        return {"score": score, "reasoning": parsed.get("reasoning", "")}
    except Exception as e:
        return {"score": 0.0, "reasoning": f"Parse error: {e} | Raw: {raw[:100]}"}


def evaluate_single(item: dict) -> dict:
    """
    Evaluates one (question, contexts, answer, ground_truth) entry.
    Makes 4 NIM calls sequentially with sleep between each.
    Returns scores + reasoning for all 4 metrics.
    """
    q = item["question"]
    a = item["answer"]
    gt = item["ground_truth"]
    ctx = item.get("contexts", [])

    # Build context strings for prompts
    context_joined = "\n\n".join(ctx) if ctx else "[No context retrieved]"
    context_numbered = (
        "\n\n".join(f"[Chunk {i + 1}]:\n{c}" for i, c in enumerate(ctx))
        if ctx
        else "[No context retrieved]"
    )

    scores = {}

    # 1. Faithfulness
    raw = nim_call(
        FAITHFULNESS_PROMPT.format(question=q, context=context_joined, answer=a),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["faithfulness"] = safe_parse_score(raw)

    # 2. Answer Relevancy
    raw = nim_call(
        ANSWER_RELEVANCY_PROMPT.format(question=q, answer=a),
        model=JUDGE_MODEL,
        max_tokens=300,
    )
    scores["answer_relevancy"] = safe_parse_score(raw)

    # 3. Context Precision
    raw = nim_call(
        CONTEXT_PRECISION_PROMPT.format(
            question=q, ground_truth=gt, context_numbered=context_numbered
        ),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["context_precision"] = safe_parse_score(raw)

    # 4. Context Recall
    raw = nim_call(
        CONTEXT_RECALL_PROMPT.format(
            question=q, ground_truth=gt, context=context_joined
        ),
        model=JUDGE_MODEL,
        max_tokens=400,
    )
    scores["context_recall"] = safe_parse_score(raw)

    return scores


def evaluate_pipeline(pipeline_outputs: list[dict], strategy_name: str) -> list[dict]:
    """
    Evaluates all outputs of one RAG strategy.
    Returns a list of per-question results with scores.
    """
    results = []
    total = len(pipeline_outputs)
    print(f"\n{'=' * 60}")
    print(
        f"Evaluating: {strategy_name} ({total} questions × 4 metrics = {total * 4} NIM calls)"
    )
    print(f"{'=' * 60}")

    for i, item in enumerate(pipeline_outputs, 1):
        print(f"\n[{i}/{total}] {item['question'][:65]}...")

        # Skip entries that errored during pipeline run
        if item["answer"].startswith("ERROR") or item["answer"] == "TODO":
            print("  ⏭ Skipped (pipeline error)")
            continue

        scores = evaluate_single(item)

        result = {
            "question": item["question"],
            "answer": item["answer"],
            "ground_truth": item["ground_truth"],
            "n_contexts": len(item.get("contexts", [])),
            "faithfulness": scores["faithfulness"]["score"],
            "answer_relevancy": scores["answer_relevancy"]["score"],
            "context_precision": scores["context_precision"]["score"],
            "context_recall": scores["context_recall"]["score"],
            "reasoning": scores,
        }
        results.append(result)

        # Print live scores
        print(
            f"  F={result['faithfulness']:.2f}  "
            f"AR={result['answer_relevancy']:.2f}  "
            f"CP={result['context_precision']:.2f}  "
            f"CR={result['context_recall']:.2f}"
        )

    print(f"\n✅ Done: {len(results)}/{total} questions evaluated")
    return results


print("✅ Evaluator functions defined.")


✅ Evaluator functions defined.


In [31]:
# ── Evaluate Vector RAG ───────────────────────────────────────────────────────
vector_eval_file = CACHE_DIR / "vector_rag_eval_results.json"

if vector_eval_file.exists():
    with open(vector_eval_file) as f:
        vector_eval_results = json.load(f)
    print(
        f"✅ Loaded cached Vector RAG eval results ({len(vector_eval_results)} entries)"
    )
else:
    vector_eval_results = evaluate_pipeline(vector_rag_outputs, "Vector RAG (ChromaDB)")
    with open(vector_eval_file, "w") as f:
        json.dump(vector_eval_results, f, indent=2)
    print(f"✅ Saved to {vector_eval_file}")


✅ Loaded cached Vector RAG eval results (29 entries)


In [32]:
with open("../cache/vector_rag_eval_results.json", "r", encoding="utf-8") as f:
    output_set = json.load(f)

print(len(output_set))


29


In [33]:
import pandas as pd

METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]


def compute_averages(results: list[dict], strategy_name: str) -> dict:
    if not results:
        return {"Strategy": strategy_name, **{m: None for m in METRICS}}
    df = pd.DataFrame(results)
    avgs = df[METRICS].mean().round(3)
    return {"Strategy": strategy_name, **avgs.to_dict()}


# Build comparison table
summary_rows = [
    compute_averages(vector_eval_results, "Vector RAG (ChromaDB)"),
]

summary_df = pd.DataFrame(summary_rows).set_index("Strategy")

print("\n" + "=" * 65)
print("  COMPARATIVE RAG EVALUATION RESULTS")
print("=" * 65)
print(summary_df.to_string())
print("=" * 65)
print("  All scores 0.0 – 1.0   |   Higher is better")
print(
    "  F=Faithfulness | AR=Answer Relevancy | CP=Context Precision | CR=Context Recall"
)



  COMPARATIVE RAG EVALUATION RESULTS
                       faithfulness  answer_relevancy  context_precision  context_recall
Strategy                                                                                
Vector RAG (ChromaDB)         0.966             0.914              0.503           0.793
  All scores 0.0 – 1.0   |   Higher is better
  F=Faithfulness | AR=Answer Relevancy | CP=Context Precision | CR=Context Recall
